In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras import Model, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/face-shape-new"

TRAIN_DIR = os.path.join(DATASET_PATH, "training")
VAL_DIR   = os.path.join(DATASET_PATH, "testing_set2")

MODEL_SAVE_PATH = "/content/drive/MyDrive/face_shape_best_model.keras"

BATCH_SIZE = 32
TARGET_SIZE = (224, 224)
INPUT_SHAPE = (224, 224, 3)
LEARNING_RATE = 1e-5
NUM_CLASSES = 5

In [ ]:
def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'r', label='Training Accuracy')
    plt.plot(epochs, val_acc, 'b', label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'r', label='Training Loss')
    plt.plot(epochs, val_loss, 'b', label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
def evaluate_model_predictions(model, generator, class_indices, title="Confusion Matrix"):
    predicted_probs = model.predict(generator, verbose=1)
    predicted_classes = np.argmax(predicted_probs, axis=1)

    true_classes = generator.classes

    if len(true_classes) != len(predicted_classes):
        print("Mismatch in sample count.")
        return None

    conf_matrix = confusion_matrix(true_classes, predicted_classes)

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        conf_matrix,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_indices.keys(),
        yticklabels=class_indices.keys()
    )
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.show()

    return conf_matrix

In [ ]:
def generate_classification_report(model, generator, class_indices):
    predicted_probs = model.predict(generator, verbose=1)
    predicted_classes = np.argmax(predicted_probs, axis=1)

    true_classes = generator.classes

    if len(true_classes) != len(predicted_classes):
        print("Mismatch in sample count.")
        return None

    report = classification_report(
        true_classes,
        predicted_classes,
        target_names=list(class_indices.keys())
    )
    return report

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    verbose=1,
    min_lr=1e-6
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    filepath=MODEL_SAVE_PATH,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

validation_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("Class indices:", train_generator.class_indices)
print("Num classes:", train_generator.num_classes)

In [ ]:
vgg16_model = VGG16(
    input_shape=INPUT_SHAPE,
    include_top=False,
    weights='imagenet',
    pooling='max'
)

for layer in vgg16_model.layers[:-4]:
    layer.trainable = False

for layer in vgg16_model.layers[-12:]:
    layer.trainable = True

x = Flatten()(vgg16_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=vgg16_model.input, outputs=x)

model.compile(
    optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=50,
    verbose=1,
    callbacks=[early_stopping, reduce_lr, checkpoint]
)

In [ ]:
plot_history(history)

In [ ]:
best_model = tf.keras.models.load_model(MODEL_SAVE_PATH)
print("✅ Best model loaded from Drive")

In [ ]:
best_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/face_shape_best_model.keras"
)

best_model.save(
    "/content/drive/MyDrive/face_shape_final.h5",
    save_format='h5'
)

print("Done - download face_shape_final.h5")

In [ ]:
val_probs = best_model.predict(validation_generator, verbose=1)
val_pred_classes = np.argmax(val_probs, axis=1)
true_classes = validation_generator.classes

print("Validation Accuracy:", accuracy_score(true_classes, val_pred_classes))
print(generate_classification_report(best_model, validation_generator, train_generator.class_indices))

In [ ]:
evaluate_model_predictions(
    best_model,
    validation_generator,
    train_generator.class_indices,
    title="VGG16 Confusion Matrix"
)